## 1. Check GPU, RAM, and Disk Space

Default: **Q3 DiT + 4B INT8 encoder**. If VRAM is constrained, switch `QUANT = "Q2"` and load the model in cell 3. On high system RAM instances, you may select `full_32b_q2` to evaluate quality differences (~8.5 GB extra download).

In [ ]:
import os, sys, shutil, subprocess, pathlib, time

# @title Model Weights Configuration { display-mode: "form" }
QUANT = "Q4"  # @param ["Q2", "Q3", "Q4"]
TEXT_ENCODER = "compact_4b_int8"  # @param ["compact_4b_int8", "full_32b_q2"]
MAX_VRAM_GB = 12  # @param {type:"slider", min:6, max:14, step:1}

ROOT = pathlib.Path("/content")
MODELS_DIR = ROOT / "h3_models"
OUTPUT_DIR = ROOT / "h3_outputs"
SRC_DIR = ROOT / "stable-diffusion.cpp"
for directory in (MODELS_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

try:
    import torch
    assert torch.cuda.is_available(), "GPU not detected! Go to Runtime → Change runtime type → T4 GPU."
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    ram_gib = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 1024**3
    disk_free_gib = shutil.disk_usage(ROOT).free / 1024**3
    print(f"GPU: {name} | compute {major}.{minor} | VRAM {vram_gib:.1f} GiB")
    print(f"Host RAM: {ram_gib:.1f} GiB | Free Disk: {disk_free_gib:.1f} GiB")
    if disk_free_gib < 25:
        raise RuntimeError("Requires at least ~25 GiB free disk space. Clean up files or change runtime.")
    if ram_gib < 14:
        print("⚠️ Standard Colab RAM: using offload/streaming; try Q2 and 1024² first to avoid OOM.")
    print(f"Selected DiT {QUANT}, encoder {TEXT_ENCODER}, GPU VRAM limit {MAX_VRAM_GB} GiB")
except ImportError as exc:
    raise RuntimeError("This notebook is intended for Google Colab GPU with PyTorch pre-installed.") from exc

## 2. Install Dependencies & Build `stable-diffusion.cpp` for T4

Initial compilation takes a few minutes. Compiles native CUDA (`sm_75` for Tesla T4), **without heavy Diffusers frameworks**. Automatically skips clone/build if the binary already exists.

```
import shutil, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.34,<2", "ninja", "pillow"], check=True)
nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
if not pathlib.Path(nvcc).exists():
    raise RuntimeError("nvcc not found. Please use a Colab GPU runtime with CUDA Toolkit.")

sd_cli = SRC_DIR / "build" / "bin" / "sd-cli"
if not sd_cli.exists():
    if not (SRC_DIR / "CMakeLists.txt").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--recurse-submodules", "--shallow-submodules",
            "https://github.com/leejet/stable-diffusion.cpp.git", str(SRC_DIR)
        ], check=True)
    subprocess.run(["git", "submodule", "update", "--init", "--recursive", "--depth", "1"], cwd=SRC_DIR, check=True)
    arch = f"{major}{minor}"
    subprocess.run([
        "cmake", "-S", str(SRC_DIR), "-B", str(SRC_DIR / "build"), "-G", "Ninja",
        "-DCMAKE_BUILD_TYPE=Release", "-DSD_CUDA=ON",
        f"-DCMAKE_CUDA_ARCHITECTURES={arch}",
        "-DSD_CUDA_CUTLASS_PATH:PATH=", "-DSD_WEBP=OFF", "-DSD_WEBM=OFF"
    ], check=True)
    subprocess.run(["cmake", "--build", str(SRC_DIR / "build"), "-j", "2"], check=True)

assert sd_cli.exists(), f"Binary not found after build: {sd_cli}"
subprocess.run([str(sd_cli), "--version"], check=True)
print("✅ CUDA build successful:", sd_cli)
```



## Backup build

```
from google.colab import drive
drive.mount('/content/drive')

!tar -czf /content/drive/MyDrive/sd_cpp_cuda_t4.tar.gz \
    -C /content/stable-diffusion.cpp/build bin
```



## Restore build

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/stable-diffusion.cpp/build

!tar -xzf /content/drive/MyDrive/sd_cpp_cuda_t4.tar.gz \
    -C /content/stable-diffusion.cpp/build

!/content/stable-diffusion.cpp/build/bin/sd-cli --version

In [ ]:
sd_cli = "/content/stable-diffusion.cpp/build/bin/sd-cli"

## 3. Download Selected Quantized Weights

Downloads from Hugging Face directly into `/content/h3_models`, reusable from session cache. **Avoids downloading full repositories** since original repos contain numerous massive variants.

In [ ]:
# @title
from huggingface_hub import hf_hub_download

DIT_FILES = {
    "Q2": "MiniMax-H3-Image-AD-Q2_K.gguf",
    "Q3": "MiniMax-H3-Image-AD-Q3_K.gguf",
    "Q4": "MiniMax-H3-Image-AD-Q4_K.gguf",
}
ENCODERS = {
    "compact_4b_int8": (
        "PrunaAI/P-H3-Image-Comfy",
        "text_encoders/p-h3-image_text_encoder_int8_convrot.safetensors",
    ),
    "full_32b_q2": (
        "realrebelai/MiniMax-H3_GGUFs",
        "qwen3vl-32B-MiniMax-H3-Q2_K.gguf",
    ),
}

specs = {
    "dit": ("AtomicChat/MiniMax-H3-Image-GGUF", DIT_FILES[QUANT]),
    "encoder": ENCODERS[TEXT_ENCODER],
    "vae": ("Comfy-Org/MiniMax-H3-Image", "vae/minimax_h3_image_vae_fp16.safetensors"),
}
MODEL_PATHS = {}
for label, (repo, filename) in specs.items():
    print(f"\n📥 {label}: {repo}/{filename}")
    path = hf_hub_download(repo_id=repo, filename=filename, local_dir=str(MODELS_DIR))
    MODEL_PATHS[label] = pathlib.Path(path)
    print(f"✅ {MODEL_PATHS[label]} ({MODEL_PATHS[label].stat().st_size / 1e9:.2f} GB)")

print("\nTotal size:", round(sum(p.stat().st_size for p in MODEL_PATHS.values()) / 1e9, 2), "GB")

## 4. Image Generation (Text-to-Image)

Write detailed English prompts for optimal composition adherence. Default test resolution: **1024×1024** for quick validation; scale up to `1536×1536` or `2048×2048` afterwards. Keep `STEPS=6`, `CFG=1.0`, `euler`, and `sigmas=minimax-h3-image` as this is the distilled sampling schedule.

In [ ]:
from datetime import datetime
from PIL import Image
from IPython.display import display

# @title Generate Image { display-mode: "form" }
PROMPT = "A photorealistic professional fashion photograph of an adult East Asian woman in her mid-twenties sitting gracefully on the beige stone edge of a luxurious outdoor swimming pool on a bright summer afternoon.  She has long, naturally flowing dark brown hair, a delicate oval face, realistic skin texture with subtle pores, soft natural makeup, and warm brown eyes. Both eyes have anatomically natural shapes, aligned pupils, realistic iris patterns, subtle reflections of daylight, and clearly defined eyelashes. Her expression is relaxed, with a gentle, natural smile.  She wears an elegant floral-patterned one-piece swimsuit. Her posture is comfortable and anatomically natural, with her hands resting gently near her knees. Her head is turned slightly toward the camera, making direct eye contact.  The swimming pool has crystal-clear turquoise water with realistic sunlight reflections. Behind her are green tropical plants, neatly trimmed hedges, white lounge chairs, and a modern resort building.  The composition is a medium portrait with the woman occupying approximately 65% of the image height. Her face is clearly visible and sharply focused. The camera is at eye level, using a natural 50mm photographic perspective.  Soft directional sunlight illuminates her face evenly. Natural skin tones, realistic dynamic range, fine photographic detail, subtle background depth of field, and authentic editorial photography aesthetics. The image should look like a genuine photograph captured with a professional full-frame camera." # @param {type:"string"}
WIDTH = 1920  # @param {type:"integer"}
HEIGHT = 1088  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
USE_FLASH_ATTENTION = True  # @param {type:"boolean"}

assert WIDTH % 32 == 0 and HEIGHT % 32 == 0, "Width and height must be divisible by 32."
assert WIDTH >= 512 and HEIGHT >= 512, "Resolution should be at least 512px."
assert MAX_VRAM_GB < vram_gib, "Set MAX_VRAM_GB lower than actual physical VRAM."

def build_command(prompt, width, height, seed, output_path, references=()):
    args = [
        str(sd_cli),
        "--diffusion-model", str(MODEL_PATHS["dit"]),
        "--llm", str(MODEL_PATHS["encoder"]),
        "--vae", str(MODEL_PATHS["vae"]),
        "-p", prompt,
        "-W", str(width), "-H", str(height),
        "--seed", str(seed),
        "--cfg-scale", "1.0", "--steps", "6",
        "--sampling-method", "euler", "--sigmas", "minimax-h3-image",
        "--rng", "cpu", "--auto-fit", "on",
        "--max-vram", str(MAX_VRAM_GB), "--vae-tiling",
        "-o", str(output_path),
    ]
    if USE_FLASH_ATTENTION:
        args += ["--diffusion-fa"]
    for reference in references:
        args += ["--ref-image", str(reference)]
    return args

out_path = OUTPUT_DIR / f"h3_{QUANT.lower()}_{WIDTH}x{HEIGHT}_seed{SEED}_{datetime.now():%Y%m%d_%H%M%S}.png"
started = time.monotonic()
cmd = build_command(PROMPT, WIDTH, HEIGHT, SEED, out_path)
print("🎨 Starting image generation... (if low on RAM, try Q2 / 1024² or increase runtime RAM)")
try:
    subprocess.run(cmd, cwd=SRC_DIR, check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        "Image generation failed. Check errors above. If OOM: try Q2, WIDTH=HEIGHT=1024, "
        "MAX_VRAM_GB=10; if CUDA INT8 issues on T4, set TEXT_ENCODER="
        "'full_32b_q2' (requires more system RAM) and re-download weights."
    ) from exc
assert out_path.exists(), f"Process exited without creating output file: {out_path}"
print(f"✅ Completed in {time.monotonic()-started:.1f}s: {out_path}")
display(Image.open(out_path))

## 5. (Optional) Image Editing with MiniMax H3-Image

Run this cell to edit an existing image. Colab lets you upload a reference image; reference it in the prompt as `<Picture 1>`.

In [ ]:
# @title Image Editing (run only when needed) { display-mode: "form" }
EDIT_PROMPT = "Edit <Picture 1>. Keep the overall composition and piano unchanged, but replace the morning light with a beautiful blue-hour atmosphere and glowing lanterns."  # @param {type:"string"}
EDIT_WIDTH = 1920  # @param {type:"integer"}
EDIT_HEIGHT = 1088  # @param {type:"integer"}

from google.colab import files
uploads = files.upload()
if not uploads:
    print("No image selected, skipping.")
else:
    image_name = next(iter(uploads))
    reference_path = ROOT / pathlib.Path(image_name).name
    reference_path.write_bytes(uploads[image_name])
    edited_path = OUTPUT_DIR / f"h3_edit_{SEED}_{datetime.now():%Y%m%d_%H%M%S}.png"
    edit_cmd = build_command(EDIT_PROMPT, EDIT_WIDTH, EDIT_HEIGHT, SEED, edited_path, [reference_path])
    subprocess.run(edit_cmd, cwd=SRC_DIR, check=True)
    print("✅ Edited image:", edited_path)
    display(Image.open(edited_path))